<div align="center">

![Course](https://img.shields.io/badge/Course-Advanced%20Data%20Science%20Lab-blue?style=for-the-badge&logo=python&logoColor=white)
![Code](https://img.shields.io/badge/Code-MCA33PE21-informational?style=for-the-badge)
![Assignment](https://img.shields.io/badge/Assignment-01-success?style=for-the-badge)

---

# Advanced Data Science Lab [MCA33PE21]
## Formative Assessment – 01
### Installation, Python Programming and Statistical Concepts

---
<br>

| Field | Details |
|---|---|
| **Academic Year** | **2026-2027** |
| **Class / Division** | **SYMCA (Semester-I)** |
| **PRN** | 125M1H026 |
| **Student Name** | Shantanu Suryawanshi |
| **Submission Date** | 02-09-206 |
</div>

# 🏠 Log Transformation & Scaling

**Dataset:** Ames Housing / House Prices  
**Target:** `SalePrice`

## 🎯 Objective
Study the effect of log transformation and feature scaling on a simple Linear Regression model.

### Required workflow
- Load and explore data
- Visualize distributions, relationships and outliers
- Handle suitable extreme values
- Split into training/testing sets
- Train Linear Regression
- Evaluate using **R², MAE, MSE and RMSE**
- Visualize a regression line
- Repeat the pipeline after log transformation + StandardScaler
- Compare both experiments
- Save outputs for a Streamlit dashboard

> The final conclusion must be based on measured results; improvement must not be assumed.

# 🧰 1. Import Libraries
We use pandas for data handling, NumPy for numerical operations, Matplotlib for visualization, and scikit-learn for preprocessing, regression and evaluation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)
os.makedirs("../outputs", exist_ok=True)

# 📂 2. Load the Dataset
The notebook is inside `notebooks/`, so the Kaggle training file is loaded from `../data/train.csv`.

In [ ]:
data_file = "../data/train.csv"
df = pd.read_csv(data_file)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# 🔎 3. Explore the Dataset
First inspect records, dimensions, data types, missing values and descriptive statistics.

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0].head(20)

In [ ]:
df.describe().T

# 🎯 4. Select Features and Target
`SalePrice` is the regression target. We use a maximum of ten meaningful numerical predictors so the experiment remains interpretable.

The final transformation decision is based on the observed distributions rather than assumptions.

In [ ]:
candidate_features = [
    "OverallQual", "GrLivArea", "GarageCars", "GarageArea",
    "TotalBsmtSF", "1stFlrSF", "FullBath", "YearBuilt",
    "YearRemodAdd", "TotRmsAbvGrd"
]
target_column = "SalePrice"

model_data = df[candidate_features + [target_column]].copy()
model_data.head()

# 🧹 5. Handle Missing Values
For numerical predictors, median imputation is robust to extreme values. Rows with a missing target are removed because they cannot provide a training label.

In [ ]:
print(model_data[candidate_features].isnull().sum()[lambda x: x > 0])

for column in candidate_features:
    if model_data[column].isnull().any():
        model_data[column] = model_data[column].fillna(model_data[column].median())

model_data = model_data.dropna(subset=[target_column]).copy()

print("Remaining rows:", len(model_data))
print("Remaining missing values:", model_data.isnull().sum().sum())

# 📈 6. Measure Skewness
Skewness provides numerical evidence of asymmetric distributions. Histograms below provide the corresponding visual evidence.

In [ ]:
skewness_before = model_data[candidate_features + [target_column]].skew().sort_values(ascending=False)
skewness_before.to_frame("Skewness")

In [ ]:
plot_columns = ["SalePrice", "GrLivArea", "TotalBsmtSF", "1stFlrSF"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, column in zip(axes.ravel(), plot_columns):
    axis.hist(model_data[column], bins=30, edgecolor="black")
    axis.set_title(f"Distribution of {column}")
    axis.set_xlabel(column)
    axis.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../outputs/original_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# 📦 7. Box Plots and Outliers
The IQR rule identifies potential outliers. A potential outlier is not automatically an incorrect observation; in housing data, unusually large houses can be legitimate.

In [ ]:
plt.figure(figsize=(12, 6))
model_data[candidate_features + [target_column]].boxplot(rot=45)
plt.title("Box Plot of Selected Variables")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("../outputs/boxplots_before_outlier_handling.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def count_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = pd.Series({
    column: count_iqr_outliers(model_data[column])
    for column in candidate_features + [target_column]
}).sort_values(ascending=False)

outlier_counts.to_frame("Potential IQR Outliers")

# 🧠 8. Outlier Treatment
We use IQR capping for selected size-related variables. Capping retains the observations while limiting the influence of extreme magnitudes.

We do **not** automatically delete every outlier because extreme house characteristics may be valid observations.

In [ ]:
def iqr_cap(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)

clean_data = model_data.copy()
cap_columns = ["GrLivArea", "TotalBsmtSF", "1stFlrSF"]

for column in cap_columns:
    clean_data[column] = iqr_cap(clean_data[column])

outlier_counts_after = pd.Series({
    column: count_iqr_outliers(clean_data[column])
    for column in candidate_features + [target_column]
}).sort_values(ascending=False)

pd.DataFrame({"Before": outlier_counts, "After": outlier_counts_after})

In [ ]:
plt.figure(figsize=(12, 6))
clean_data[candidate_features + [target_column]].boxplot(rot=45)
plt.title("Box Plot After Selected IQR Capping")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("../outputs/boxplots_after_outlier_handling.png", dpi=150, bbox_inches="tight")
plt.show()

# 🔗 9. Feature Comparison with SalePrice
A correlation view helps us understand which selected variables have stronger linear relationships with the target. Correlation is exploratory evidence, not proof of causation.

In [ ]:
correlation_with_target = (
    clean_data[candidate_features + [target_column]]
    .corr()[target_column]
    .drop(target_column)
    .sort_values(key=np.abs, ascending=False)
)
correlation_with_target.to_frame("Correlation with SalePrice")

In [ ]:
plt.figure(figsize=(10, 6))
correlation_with_target.sort_values().plot(kind="barh")
plt.title("Feature Correlation with SalePrice")
plt.xlabel("Correlation")
plt.tight_layout()
plt.savefig("../outputs/feature_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

# ✂️ 10. Train/Test Split — Baseline
We use the same 80/20 split for both experiments so that the comparison is fair and reproducible.

In [ ]:
X = clean_data[candidate_features].copy()
y = clean_data[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

# 🤖 11. Train Baseline Linear Regression
The baseline model uses the selected features without log transformation or scaling.

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

# 📏 12. Evaluate Baseline Model
Required metrics: R², MAE, MSE and RMSE.

In [ ]:
baseline_r2 = r2_score(y_test, baseline_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(baseline_mse)

baseline_results = pd.DataFrame({
    "Metric": ["R²", "MAE", "MSE", "RMSE"],
    "Before Transformation": [
        baseline_r2, baseline_mae, baseline_mse, baseline_rmse
    ]
})
baseline_results

# 📈 13. Baseline Actual vs Predicted
For a multiple regression model, there is no single 2D regression line. This actual-vs-predicted plot is therefore the primary model visualization.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, baseline_predictions, alpha=0.6)

low = min(y_test.min(), baseline_predictions.min())
high = max(y_test.max(), baseline_predictions.max())

plt.plot([low, high], [low, high], linestyle="--")
plt.title("Baseline: Actual vs Predicted SalePrice")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.tight_layout()
plt.savefig("../outputs/baseline_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

# 📐 14. Regression Line Visualization
Because a multiple regression model has many dimensions, we use the strongest correlated feature for a simple 2D regression-line visualization. This is an explanatory plot, separate from the full baseline model.

In [ ]:
main_feature = correlation_with_target.abs().idxmax()

line_model = LinearRegression()
line_model.fit(clean_data[[main_feature]], clean_data[target_column])

line_x = np.linspace(
    clean_data[main_feature].min(),
    clean_data[main_feature].max(),
    200
).reshape(-1, 1)
line_y = line_model.predict(line_x)

plt.figure(figsize=(9, 6))
plt.scatter(clean_data[main_feature], clean_data[target_column], alpha=0.35)
plt.plot(line_x, line_y, linewidth=2)
plt.title(f"Regression Line: {main_feature} vs SalePrice")
plt.xlabel(main_feature)
plt.ylabel("SalePrice")
plt.tight_layout()
plt.savefig("../outputs/regression_line.png", dpi=150, bbox_inches="tight")
plt.show()

print("Feature used:", main_feature)

# 🔄 15. Log Transformation
The transformed experiment applies `log1p(x) = log(1 + x)` to selected positive, skewed predictors.

The target remains on the original `SalePrice` scale so both experiments can be compared in the same units.

In [ ]:
transformed_data = clean_data.copy()

log_columns = ["GrLivArea", "TotalBsmtSF", "1stFlrSF"]
log_columns = [column for column in log_columns if column in transformed_data.columns]

for column in log_columns:
    transformed_data[column] = np.log1p(transformed_data[column])

print("Log-transformed columns:", log_columns)

# 📊 16. Compare Skewness Before and After

In [ ]:
skewness_after = transformed_data[candidate_features].skew()

skewness_comparison = pd.DataFrame({
    "Before": clean_data[candidate_features].skew(),
    "After": skewness_after
})

skewness_comparison["Absolute Skewness Change"] = (
    skewness_comparison["Before"].abs()
    - skewness_comparison["After"].abs()
)

skewness_comparison.sort_values("Absolute Skewness Change", ascending=False)

In [ ]:
comparison_feature = log_columns[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(clean_data[comparison_feature], bins=30, edgecolor="black")
axes[0].set_title(f"Before: {comparison_feature}")
axes[0].set_xlabel(comparison_feature)
axes[0].set_ylabel("Frequency")

axes[1].hist(transformed_data[comparison_feature], bins=30, edgecolor="black")
axes[1].set_title(f"After: log1p({comparison_feature})")
axes[1].set_xlabel(f"log1p({comparison_feature})")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../outputs/before_and_after_log.png", dpi=150, bbox_inches="tight")
plt.show()

# 📏 17. StandardScaler
The scaler is fitted **only on the training data** and then reused on the test data. This prevents test-set information from leaking into model training.

In [ ]:
X_transformed = transformed_data[candidate_features]
y_transformed = transformed_data[target_column]

X_train_transformed = X_transformed.loc[X_train.index]
X_test_transformed = X_transformed.loc[X_test.index]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

# 🤖 18. Train Transformed Linear Regression

In [ ]:
transformed_model = LinearRegression()
transformed_model.fit(X_train_scaled, y_train)
transformed_predictions = transformed_model.predict(X_test_scaled)

# 📏 19. Evaluate Transformed Model

In [ ]:
transformed_r2 = r2_score(y_test, transformed_predictions)
transformed_mae = mean_absolute_error(y_test, transformed_predictions)
transformed_mse = mean_squared_error(y_test, transformed_predictions)
transformed_rmse = np.sqrt(transformed_mse)

transformed_results = pd.DataFrame({
    "Metric": ["R²", "MAE", "MSE", "RMSE"],
    "After Log + Scaling": [
        transformed_r2, transformed_mae, transformed_mse, transformed_rmse
    ]
})
transformed_results

# 📈 20. Transformed Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, transformed_predictions, alpha=0.6)

low = min(y_test.min(), transformed_predictions.min())
high = max(y_test.max(), transformed_predictions.max())

plt.plot([low, high], [low, high], linestyle="--")
plt.title("After Log + Scaling: Actual vs Predicted SalePrice")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.tight_layout()
plt.savefig("../outputs/transformed_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

# ⚖️ 21. Compare Both Models
Both experiments use the same observations, target, split and evaluation metrics. The comparison isolates the effect of the preprocessing pipeline.

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["R²", "MAE", "MSE", "RMSE"],
    "Before Transformation": [
        baseline_r2, baseline_mae, baseline_mse, baseline_rmse
    ],
    "After Log + Scaling": [
        transformed_r2, transformed_mae, transformed_mse, transformed_rmse
    ]
})

comparison.to_csv("../outputs/model_comparison.csv", index=False)
comparison

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(len(comparison["Metric"]))
width = 0.35

plt.bar(x - width/2, comparison["Before Transformation"], width,
        label="Before Transformation")
plt.bar(x + width/2, comparison["After Log + Scaling"], width,
        label="After Log + Scaling")

plt.xticks(x, comparison["Metric"])
plt.title("Regression Model Performance Comparison")
plt.ylabel("Metric Value")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# 🧮 22. Percentage Change
For MAE, MSE and RMSE, a negative percentage change means the error decreased. For R², a positive change generally indicates improvement.

In [ ]:
summary = comparison.copy()
summary["Change (%)"] = (
    (summary["After Log + Scaling"] - summary["Before Transformation"])
    / summary["Before Transformation"].replace(0, np.nan)
) * 100
summary

# 📊 26. Final Model Accuracy Comparison

The following comparison shows the performance of both regression models using the **R² score**.

- **Before Transformation:** Linear Regression using the original feature values.
- **After Log + Scaling:** Linear Regression using log-transformed features followed by StandardScaler.

A higher R² score indicates that the model explains a larger proportion of the variation in `SalePrice`.

The graph provides a direct visual comparison of both model results.

In [ ]:
# Display the R² score of both models
print("==============================================")
print("       LINEAR REGRESSION MODEL ACCURACY")
print("==============================================")

print(f"Before Transformation : {baseline_r2:.4f}")
print(f"After Log + Scaling   : {transformed_r2:.4f}")

if transformed_r2 > baseline_r2:
    print("\nResult: After Log + Scaling performed better.")
elif transformed_r2 < baseline_r2:
    print("\nResult: Before Transformation performed better.")
else:
    print("\nResult: Both models have the same R² score.")

# Draw a simple comparison graph
model_names = [
    "Before Transformation",
    "After Log + Scaling"
]

r2_scores = [
    baseline_r2,
    transformed_r2
]

plt.figure(figsize=(8, 5))

bars = plt.bar(model_names, r2_scores)

plt.title("R² Score Comparison of Both Models")
plt.xlabel("Model")
plt.ylabel("R² Score")
plt.ylim(0, 1)

# Display the exact R² value above each bar
for bar, score in zip(bars, r2_scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{score:.4f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig("../outputs/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Selected features:")
print(candidate_features)

print("\nLog-transformed features:")
print(log_columns)

print("\nMain feature used for regression-line visualization:")
print(main_feature)

print("\nFinal model comparison:")
display(comparison)

# ✅ Experiment Complete

```text
Load Data
   ↓
Explore Data
   ↓
Select ≤ 10 Features
   ↓
Handle Missing Values
   ↓
Visualize + Measure Skewness
   ↓
Inspect + Treat Suitable Outliers
   ↓
Train/Test Split
   ↓
Baseline Linear Regression
   ↓
R² + MAE + MSE + RMSE
   ↓
Log Transformation
   ↓
StandardScaler
   ↓
Second Linear Regression
   ↓
R² + MAE + MSE + RMSE
   ↓
Compare
   ↓
Conclusion
```

---

<div align="center">

### Declaration

_I hereby declare that the work submitted in this practical assignment is my own and has not been copied from any other source._

| | |
|---|---|
| **Student Name** | Shantanu Suryawanshi |
| **PRN** | 125M1H026 |
| **Date** | 02-09-2026 |

---

**Prof. Prakash Ukhalkar**  
Course Teacher - Advanced Data Science Lab [MCA33PE21]

![](https://img.shields.io/badge/MCA33PE21-Advanced%20Data%20Science%20Lab-blue?style=flat-square)
![](https://img.shields.io/badge/Practical%20Assignment-01-green?style=flat-square)

</div>